# NIDS Master Research Pipeline (Comprehensive)

Notebook ini memecah pipeline jadi langkah detail dari awal sampai akhir, mengikuti pola repo referensi (`Intrusion-Detection-Pipeline` dan `Intrusion-Detection-CICIDS2017`) tapi tetap kompatibel dengan implementasi project ini.

## Tujuan
- Menjalankan proses riset secara bertahap: setup -> audit data -> preprocess -> training -> evaluasi -> baseline -> ringkasan.
- Menutup gap dari notebook lama yang terlalu ringkas.
- Menyediakan checkpoint evaluasi yang mudah diverifikasi.


## Colab Bootstrap (Colab-only)

Section ini hanya aktif saat runtime Google Colab.
Fungsinya: mount Drive, clone/pull branch target, optional symlink data dari Drive, setup `kaggle.json`,
dan symlink semua output artifact (`data/research`, `models/research`, `results/research`, `reports/research`) ke Drive.


In [ ]:
# Colab bootstrap (Colab-only): mount drive + clone/pull repo + data/output links + kaggle token.
from pathlib import Path
import os
import shutil
import subprocess
import sys


COLAB_BOOTSTRAP_ENABLE = True
COLAB_REPO_URL = "https://github.com/akwancakra/nids-cnn-lstm-autoencoder.git"
COLAB_BRANCH = "feat/sprint2-full-isolation-runner"
COLAB_REPO_DIR = Path("/content/nids-cnn-lstm-autoencoder")

COLAB_DATA_ROOT_DRIVE = Path("/content/drive/MyDrive/nids-data/raw")
COLAB_ARTIFACTS_ROOT_DRIVE = Path("/content/drive/MyDrive/nids-data/artifacts")

COLAB_LINK_RAW_FROM_DRIVE = True
COLAB_LINK_OUTPUTS_TO_DRIVE = True
COLAB_FORCE_RELINK_OUTPUTS = True

COLAB_SETUP_KAGGLE_TOKEN = True
COLAB_KAGGLE_TOKEN_DRIVE_PATH = Path("/content/drive/MyDrive/kaggle.json")


def _is_colab_runtime() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False


def _run_shell(cmd: list[str], cwd: Path | None = None) -> None:
    print("[CMD]", " ".join(cmd))
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)


def _mount_drive() -> None:
    from google.colab import drive
    drive.mount("/content/drive")


def _clone_or_pull_repo() -> None:
    if not COLAB_REPO_DIR.exists():
        _run_shell(["git", "clone", COLAB_REPO_URL, str(COLAB_REPO_DIR)], cwd=Path("/content"))
    else:
        _run_shell(["git", "fetch", "--all"], cwd=COLAB_REPO_DIR)

    _run_shell(["git", "checkout", COLAB_BRANCH], cwd=COLAB_REPO_DIR)
    _run_shell(["git", "pull", "origin", COLAB_BRANCH], cwd=COLAB_REPO_DIR)


def _ensure_symlink_dir(repo_path: Path, drive_target: Path, allow_replace_existing: bool = False) -> None:
    drive_target.mkdir(parents=True, exist_ok=True)

    if repo_path.is_symlink():
        current_target = repo_path.resolve()
        if current_target == drive_target.resolve():
            print(f"[INFO] Symlink OK: {repo_path} -> {drive_target}")
            return
        repo_path.unlink(missing_ok=True)

    elif repo_path.exists():
        if allow_replace_existing:
            if repo_path.is_dir():
                shutil.rmtree(repo_path, ignore_errors=True)
            else:
                repo_path.unlink(missing_ok=True)
        else:
            # Fail-safe: do not silently delete real populated folders.
            try:
                has_contents = any(repo_path.iterdir())
            except Exception:
                has_contents = True
            if has_contents:
                raise RuntimeError(
                    f"Path already exists and is not symlink: {repo_path}. "
                    "Please move/delete it manually before linking to Drive."
                )
            if repo_path.is_dir():
                repo_path.rmdir()
            else:
                repo_path.unlink(missing_ok=True)

    repo_path.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(str(drive_target), str(repo_path), target_is_directory=True)
    print(f"[INFO] Symlink created: {repo_path} -> {drive_target}")


def _link_raw_data_from_drive() -> None:
    COLAB_DATA_ROOT_DRIVE.mkdir(parents=True, exist_ok=True)
    (COLAB_DATA_ROOT_DRIVE / "CIC-IDS2017").mkdir(parents=True, exist_ok=True)
    (COLAB_DATA_ROOT_DRIVE / "CSE-CIC-IDS2018").mkdir(parents=True, exist_ok=True)

    repo_raw = COLAB_REPO_DIR / "data" / "raw"
    if repo_raw.exists() and not repo_raw.is_symlink():
        # raw usually safe to replace in Colab runtime clone
        shutil.rmtree(repo_raw, ignore_errors=True)
    if repo_raw.is_symlink() or repo_raw.exists():
        try:
            repo_raw.unlink()
        except Exception:
            pass

    repo_raw.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(str(COLAB_DATA_ROOT_DRIVE), str(repo_raw), target_is_directory=True)
    print(f"[INFO] Symlink created: {repo_raw} -> {COLAB_DATA_ROOT_DRIVE}")


def _link_outputs_to_drive() -> None:
    mapping = {
        COLAB_REPO_DIR / "data" / "research": COLAB_ARTIFACTS_ROOT_DRIVE / "data" / "research",
        COLAB_REPO_DIR / "models" / "research": COLAB_ARTIFACTS_ROOT_DRIVE / "models" / "research",
        COLAB_REPO_DIR / "results" / "research": COLAB_ARTIFACTS_ROOT_DRIVE / "results" / "research",
        COLAB_REPO_DIR / "reports" / "research": COLAB_ARTIFACTS_ROOT_DRIVE / "reports" / "research",
    }
    for repo_path, drive_target in mapping.items():
        _ensure_symlink_dir(
            repo_path,
            drive_target,
            allow_replace_existing=COLAB_FORCE_RELINK_OUTPUTS,
        )


def _setup_kaggle_token() -> None:
    target_dir = Path("/root/.kaggle")
    target_dir.mkdir(parents=True, exist_ok=True)
    target = target_dir / "kaggle.json"

    if not COLAB_KAGGLE_TOKEN_DRIVE_PATH.exists():
        print(f"[WARN] Kaggle token not found at {COLAB_KAGGLE_TOKEN_DRIVE_PATH}. Skipping token setup.")
        return

    shutil.copy2(COLAB_KAGGLE_TOKEN_DRIVE_PATH, target)
    os.chmod(target, 0o600)
    print(f"[INFO] Kaggle token installed at {target}")


IS_COLAB_RUNTIME = _is_colab_runtime()
print(f"IS_COLAB_RUNTIME={IS_COLAB_RUNTIME}")

if IS_COLAB_RUNTIME and COLAB_BOOTSTRAP_ENABLE:
    _mount_drive()
    _clone_or_pull_repo()

    if COLAB_LINK_RAW_FROM_DRIVE:
        _link_raw_data_from_drive()

    if COLAB_LINK_OUTPUTS_TO_DRIVE:
        _link_outputs_to_drive()

    if COLAB_SETUP_KAGGLE_TOKEN:
        _setup_kaggle_token()

    os.chdir(COLAB_REPO_DIR)
    print(f"[INFO] Colab bootstrap done. cwd={Path.cwd()}")
else:
    print("[SKIP] Colab bootstrap is disabled or not running in Colab.")


In [ ]:
# Resolve project root (works for local + Colab), lalu optional install dependency.
from pathlib import Path
import os
import subprocess
import sys

CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content/nids-cnn-lstm-autoencoder"),
    Path("/content/drive/MyDrive/nids-cnn-lstm-autoencoder"),
]

PROJECT_ROOT = None
for c in CANDIDATES:
    if (c / "scripts").exists() and (c / "config.yaml").exists():
        PROJECT_ROOT = c.resolve()
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root tidak ditemukan. Pastikan notebook dijalankan dari repo nids-cnn-lstm-autoencoder.")

os.chdir(PROJECT_ROOT)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

INSTALL_DEPS = False
if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
    print("Dependencies installed.")
else:
    print("INSTALL_DEPS = False (skip install).")


In [ ]:
# Imports + helper utilitas eksekusi command dan IO.
import json
import time
import shlex
import shutil
import yaml
import textwrap
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)


def run_cmd_stream(cmd, cwd=None, check=True):
    if isinstance(cmd, str):
        cmd = shlex.split(cmd)
    start = time.time()
    print("$", " ".join(cmd))
    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True,
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    duration = time.time() - start
    print(f"[exit={proc.returncode}] duration={duration:.1f}s")
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}: {' '.join(cmd)}")
    return proc.returncode


def run_cmds_stream(cmd_list, cwd=None, check=True):
    for cmd in cmd_list:
        run_cmd_stream(cmd, cwd=cwd, check=check)


def read_json(path):
    p = Path(path)
    if not p.exists():
        return None
    with p.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_config(path="config.yaml"):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def save_config(cfg, path="config.yaml"):
    with open(path, "w", encoding="utf-8") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)

print("Helpers loaded.")


## Colab Persist Check (raw + outputs)

Verifikasi ini memastikan symlink `data/raw`, `data/research`, `models/research`, `results/research`, dan `reports/research`
mengarah ke Google Drive saat runtime Colab.


In [ ]:
paths_to_check = {
    "data/raw": PROJECT_ROOT / "data" / "raw",
    "data/research": PROJECT_ROOT / "data" / "research",
    "models/research": PROJECT_ROOT / "models" / "research",
    "results/research": PROJECT_ROOT / "results" / "research",
    "reports/research": PROJECT_ROOT / "reports" / "research",
}

rows = []
for name, p in paths_to_check.items():
    exists = p.exists() or p.is_symlink()
    is_link = p.is_symlink()
    resolved = str(p.resolve()) if exists else "<missing>"
    rows.append(
        {
            "path": name,
            "exists": exists,
            "is_symlink": is_link,
            "resolved_to": resolved,
        }
    )

persist_df = pd.DataFrame(rows)
display(persist_df)

if 'IS_COLAB_RUNTIME' in globals() and IS_COLAB_RUNTIME and COLAB_BOOTSTRAP_ENABLE:
    if COLAB_LINK_OUTPUTS_TO_DRIVE:
        must_link = ["data/research", "models/research", "results/research", "reports/research"]
        bad = [r["path"] for r in rows if r["path"] in must_link and not r["is_symlink"]]
        if bad:
            raise AssertionError(f"Expected symlink for: {bad}")
    print("Colab persist check passed.")
else:
    print("Local runtime / bootstrap disabled: informational check only.")


## Optional: Download Official CIC-IDS2017 (MachineLearningCSV)

Section ini untuk replace total dataset CIC-IDS2017 dari sumber resmi CIC.
Jika dijalankan, isi `data/raw/CIC-IDS2017` akan dihapus lalu diunduh ulang dari URL official.


In [ ]:
import tempfile
import urllib.request
import zipfile

DOWNLOAD_CIC_OFFICIAL = False
CIC_OFFICIAL_URL = "http://205.174.165.80/CICDataset/CIC-IDS-2017/Dataset/CIC-IDS-2017/CSVs/MachineLearningCSV.zip"
CIC_RAW_DIR = Path("data/raw/CIC-IDS2017")

# Canonical names without optional '.pcap_ISCX' suffix.
CIC_EXPECTED_CANONICAL = {
    "monday-workinghours.csv",
    "tuesday-workinghours.csv",
    "wednesday-workinghours.csv",
    "thursday-workinghours-morning-webattacks.csv",
    "thursday-workinghours-afternoon-infilteration.csv",
    "friday-workinghours-morning.csv",
    "friday-workinghours-afternoon-portscan.csv",
    "friday-workinghours-afternoon-ddos.csv",
}


def cic_canonical_name(file_name: str) -> str:
    x = str(file_name).strip().lower()
    x = x.replace(".pcap_iscx", "")
    x = x.replace("_", "-")
    x = x.replace(" ", "")
    x = x.replace("--", "-")
    return x


def count_file_lines(path: Path) -> int:
    total = 0
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            total += chunk.count(b"\n")
    return total

print(f"DOWNLOAD_CIC_OFFICIAL={DOWNLOAD_CIC_OFFICIAL}")
print(f"Target dir: {CIC_RAW_DIR}")
print(f"Source URL: {CIC_OFFICIAL_URL}")


In [ ]:
if not DOWNLOAD_CIC_OFFICIAL:
    print("DOWNLOAD_CIC_OFFICIAL=False -> skip official CIC download.")
else:
    tmp_dir = Path(tempfile.mkdtemp(prefix="cic2017_official_"))
    zip_path = tmp_dir / "MachineLearningCSV.zip"

    print("[1/5] Prepare destination...")
    if CIC_RAW_DIR.exists():
        for child in CIC_RAW_DIR.iterdir():
            if child.is_dir():
                shutil.rmtree(child, ignore_errors=True)
            else:
                child.unlink(missing_ok=True)
    CIC_RAW_DIR.mkdir(parents=True, exist_ok=True)

    print("[2/5] Download official zip...")
    with urllib.request.urlopen(CIC_OFFICIAL_URL) as response, zip_path.open("wb") as out_f:
        shutil.copyfileobj(response, out_f)

    print("[3/5] Extract zip...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(CIC_RAW_DIR)

    print("[4/5] Normalize extracted structure...")
    nested = CIC_RAW_DIR / "MachineLearningCVE"
    if nested.exists() and nested.is_dir():
        for csv_file in nested.rglob("*.csv"):
            target = CIC_RAW_DIR / csv_file.name
            if target.exists():
                target.unlink(missing_ok=True)
            shutil.move(str(csv_file), str(target))
        shutil.rmtree(nested, ignore_errors=True)

    print("[5/5] Cleanup temp files...")
    if zip_path.exists():
        zip_path.unlink(missing_ok=True)
    shutil.rmtree(tmp_dir, ignore_errors=True)

    csv_files = sorted(CIC_RAW_DIR.glob("*.csv"))
    print(f"Done. CSV files in target: {len(csv_files)}")
    for f in csv_files:
        print("-", f.name)


In [ ]:
cic_csv_files = sorted(CIC_RAW_DIR.glob("*.csv"))
if not cic_csv_files:
    raise FileNotFoundError(f"No CSV found under {CIC_RAW_DIR}. Run download cell first.")

actual_canonical = {cic_canonical_name(f.name) for f in cic_csv_files}
missing = sorted(CIC_EXPECTED_CANONICAL - actual_canonical)

if missing:
    raise AssertionError(f"Missing expected CIC files (canonical): {missing}")

if len(cic_csv_files) < 8:
    raise AssertionError(f"Expected at least 8 CSV files, found {len(cic_csv_files)}")

rows = []
for f in cic_csv_files:
    rows.append(
        {
            "file": f.name,
            "size_mb": round(f.stat().st_size / (1024 ** 2), 2),
            "line_count": count_file_lines(f),
            "canonical": cic_canonical_name(f.name),
        }
    )

validation_df = pd.DataFrame(rows).sort_values(by="file").reset_index(drop=True)
display(validation_df)
print("CIC official dataset validation passed.")


## Optional: Download CSE-CIC-IDS2018 from Kaggle

Section ini untuk refresh dataset CSE dari Kaggle mirror.
Default aman (`False`) agar tidak overwrite data tanpa sengaja.


In [ ]:
import os
import zipfile

CSE_ENABLE_KAGGLE_DOWNLOAD = False
CSE_OVERWRITE_EXISTING = True
CSE_KAGGLE_DATASET_SLUG = "solarmainframe/ids-intrusion-csv"
CSE_RAW_DIR = Path("data/raw/CSE-CIC-IDS2018")
CSE_TMP_DIR = Path("data/raw/_kaggle_tmp_cse")

# Pipeline saat ini pakai 3 file utama ini.
CSE_TARGET_FILES = {
    "02-14-2018.csv",
    "02-15-2018.csv",
    "02-16-2018.csv",
}
CSE_SELECT_ONLY_TARGET_DAYS = True


def _ensure_kaggle_token(project_root: Path) -> None:
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    token_target = kaggle_dir / "kaggle.json"

    token_candidates = [
        Path.cwd() / "kaggle.json",
        project_root / "kaggle.json",
        token_target,
        Path("/content/drive/MyDrive/Colab Notebooks/kaggle.json"),
        Path("/content/drive/MyDrive/kaggle.json"),
    ]

    chosen = next((x for x in token_candidates if x.exists()), None)
    if chosen is not None:
        if chosen.resolve() != token_target.resolve():
            shutil.copy2(chosen, token_target)
        os.chmod(token_target, 0o600)
        print(f"[INFO] Kaggle token file: {chosen}")
        return

    user = os.environ.get("KAGGLE_USERNAME", "").strip()
    key = os.environ.get("KAGGLE_KEY", "").strip()
    if not (user and key):
        raise FileNotFoundError(
            "Kaggle token tidak ditemukan. Sediakan kaggle.json (lokal/project/~/.kaggle) "
            "atau set env KAGGLE_USERNAME dan KAGGLE_KEY."
        )
    print("[INFO] Using Kaggle credentials from environment variables.")


print(f"CSE_ENABLE_KAGGLE_DOWNLOAD={CSE_ENABLE_KAGGLE_DOWNLOAD}")
print(f"Kaggle dataset slug: {CSE_KAGGLE_DATASET_SLUG}")
print(f"Target dir: {CSE_RAW_DIR}")


In [ ]:
if not CSE_ENABLE_KAGGLE_DOWNLOAD:
    print("CSE_ENABLE_KAGGLE_DOWNLOAD=False -> skip CSE Kaggle download.")
else:
    project_root = PROJECT_ROOT if "PROJECT_ROOT" in globals() else Path.cwd()

    # Prepare destination (replace existing when enabled).
    CSE_RAW_DIR.mkdir(parents=True, exist_ok=True)
    if CSE_OVERWRITE_EXISTING:
        for child in CSE_RAW_DIR.iterdir():
            if child.is_dir():
                shutil.rmtree(child, ignore_errors=True)
            else:
                child.unlink(missing_ok=True)

    # Ensure kaggle CLI available.
    run_cmd_stream([sys.executable, "-m", "pip", "install", "-q", "kaggle"], cwd=project_root)
    _ensure_kaggle_token(project_root)

    # Download zip.
    CSE_TMP_DIR.mkdir(parents=True, exist_ok=True)
    run_cmd_stream(
        [
            "kaggle",
            "datasets",
            "download",
            "-d",
            CSE_KAGGLE_DATASET_SLUG,
            "-p",
            str(CSE_TMP_DIR),
            "--force",
        ],
        cwd=project_root,
    )

    zips = sorted(CSE_TMP_DIR.glob("*.zip"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not zips:
        raise FileNotFoundError(f"No zip downloaded in {CSE_TMP_DIR}")
    zip_path = zips[0]
    extract_dir = CSE_TMP_DIR / (zip_path.stem + "_extract")
    if extract_dir.exists():
        shutil.rmtree(extract_dir, ignore_errors=True)
    extract_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)

    all_csvs = sorted(extract_dir.rglob("*.csv"))
    if not all_csvs:
        raise FileNotFoundError(f"No CSV found after extraction: {extract_dir}")

    copied = 0
    selected_names = set()
    for src in all_csvs:
        name = src.name
        if CSE_SELECT_ONLY_TARGET_DAYS and name not in CSE_TARGET_FILES:
            continue
        dst = CSE_RAW_DIR / name
        if dst.exists() and not CSE_OVERWRITE_EXISTING:
            continue
        shutil.copy2(src, dst)
        copied += 1
        selected_names.add(name)

    if CSE_SELECT_ONLY_TARGET_DAYS:
        missing_targets = sorted(CSE_TARGET_FILES - selected_names)
        if missing_targets:
            raise AssertionError(f"Missing target CSE files from kaggle extract: {missing_targets}")

    print(f"[DONE] Copied {copied} CSV files to {CSE_RAW_DIR}")

    # Cleanup temp artifacts.
    shutil.rmtree(CSE_TMP_DIR, ignore_errors=True)


In [ ]:
cse_csv_files = sorted(CSE_RAW_DIR.glob("*.csv"))
if not cse_csv_files:
    raise FileNotFoundError(f"No CSE CSV found under {CSE_RAW_DIR}")

rows = []
for f in cse_csv_files:
    rows.append(
        {
            "file": f.name,
            "size_mb": round(f.stat().st_size / (1024 ** 2), 2),
            "line_count": count_file_lines(f),
        }
    )

cse_validation_df = pd.DataFrame(rows).sort_values(by="file").reset_index(drop=True)
display(cse_validation_df)

if CSE_SELECT_ONLY_TARGET_DAYS:
    names = {f.name for f in cse_csv_files}
    missing = sorted(CSE_TARGET_FILES - names)
    if missing:
        raise AssertionError(f"CSE target files missing in destination: {missing}")

print("CSE Kaggle dataset validation passed.")


## 0) Run Control dan Safety (Isolated Runner)

Notebook ini tidak lagi mengubah `config.yaml` global.
Semua eksekusi eksperimen memakai `run_id` dari `research/sprint2/run_registry.yaml` dan generated config per-run.


In [ ]:
RUN_PREPROCESS = False
RUN_TRAIN_HYBRID = False
RUN_EVAL_ZERO_SHOT = False
RUN_EVAL_FEW_SHOT = False
RUN_TRAIN_BASELINE = False
RUN_EVAL_BASELINE_ZERO_SHOT = False
RUN_EVAL_BASELINE_FEW_SHOT = False
RUN_SUMMARY_ONLY = False

RUNNER_DRY_RUN = False
RUNNER_SKIP_EXISTING = True
FORCE_REGENERATE_CONFIG = False

REGISTRY_PATH = Path("research/sprint2/run_registry.yaml")
BASE_CONFIG_PATH = Path("config/base.yaml")
GENERATED_CONFIG_DIR = Path("research/sprint2/generated_configs")

RUN_ID_HYBRID_ZERO = "s2_hybrid_zero_seed42_base"
RUN_ID_HYBRID_FEW = "s2_hybrid_few_seed42_tp99_eval"
RUN_ID_BASELINE_ZERO = "s2_baseline_zero_seed42_base"
RUN_ID_BASELINE_FEW = "s2_baseline_few_seed42_tp99_eval"
INSPECT_RUN_ID = RUN_ID_HYBRID_ZERO


def load_registry():
    return load_config(str(REGISTRY_PATH))


REGISTRY = load_registry()
RUN_LOOKUP = {r["run_id"]: r for r in REGISTRY["runs"]}


def model_subdir(model_variant: str) -> str:
    return "cnn_lstm_ae" if str(model_variant).lower() == "hybrid" else "lstm_ae"


def generated_config_path(run_id: str) -> Path:
    return GENERATED_CONFIG_DIR / f"{run_id}.yaml"


def ensure_generated_config(run_id: str) -> Path:
    cfg_path = generated_config_path(run_id)
    if cfg_path.exists() and not FORCE_REGENERATE_CONFIG:
        return cfg_path

    cmd = [
        sys.executable,
        "scripts/research_sprint2.py",
        "--registry",
        str(REGISTRY_PATH),
        "--base-config",
        str(BASE_CONFIG_PATH),
        "--run-ids",
        run_id,
        "--dry-run",
        "--no-summarize",
    ]
    run_cmd_stream(cmd, cwd=PROJECT_ROOT)
    if not cfg_path.exists():
        raise FileNotFoundError(f"Generated config not found: {cfg_path}")
    return cfg_path


def resolve_model_path(run_id: str) -> Path:
    run = RUN_LOOKUP[run_id]
    if run.get("model_path"):
        return Path(run["model_path"])

    source_run_id = run.get("reuse_artifacts_from", run_id)
    variant = run.get("model_variant", "hybrid")
    return Path("models") / "research" / source_run_id / model_subdir(variant) / "best_model.keras"


def resolve_data_source_run(run_id: str) -> str:
    run = RUN_LOOKUP[run_id]
    return run.get("reuse_artifacts_from", run_id)


def run_eval_for_run_id(run_id: str):
    cfg_path = ensure_generated_config(run_id)
    run = RUN_LOOKUP[run_id]
    tag = run.get("tag", run_id)
    model_path = resolve_model_path(run_id)

    run_cmd_stream(
        [
            sys.executable,
            "scripts/eval_metrics.py",
            "--config",
            str(cfg_path),
            "--model",
            str(model_path),
            "--tag",
            tag,
        ],
        cwd=PROJECT_ROOT,
    )


def collect_metric_row(run_id: str, variant_label: str):
    run = RUN_LOOKUP[run_id]
    tag = run.get("tag", run_id)
    metrics_root = Path("results") / "research" / run_id / "metrics"

    cic = read_json(metrics_root / f"{tag}_cic_metrics.json")
    cse = read_json(metrics_root / f"{tag}_cse_metrics.json")
    gap = read_json(metrics_root / f"{tag}_generalization_gap.json")

    if not cic or not cse:
        return None

    return {
        "variant": variant_label,
        "run_id": run_id,
        "tag": tag,
        "cic_f1": cic.get("f1"),
        "cse_f1": cse.get("f1"),
        "cic_auc": cic.get("roc_auc"),
        "cse_auc": cse.get("roc_auc"),
        "cic_fpr": cic.get("fpr"),
        "cse_fpr": cse.get("fpr"),
        "f1_gap": None if not gap else gap.get("f1_gap"),
        "accuracy_gap": None if not gap else gap.get("accuracy_gap"),
        "threshold_method": None if not gap else gap.get("threshold_method"),
        "mode": None if not gap else gap.get("mode"),
    }


print("Run controls initialized (isolated runner mode).")
print(f"Registry: {REGISTRY_PATH}")
print(f"Base config: {BASE_CONFIG_PATH}")


# Deep analysis controls
RUN_DEEP_AUDIT = True
RUN_DRIFT_ANALYSIS = True
RUN_BASELINE_IMPORTANCE_ANALYSIS = True
ANALYSIS_SAMPLE_ROWS_PER_FILE = 120000
ANALYSIS_MAX_FILES_PER_DATASET = 10
ANALYSIS_COMPUTE_FULL_HASH = False
ANALYSIS_RANDOM_SEED = 42
ANALYSIS_FEATURE_TOP_K = 20

ANALYSIS_OUTPUT_DIR = Path("reports") / "research" / INSPECT_RUN_ID / "analysis"
ANALYSIS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Deep analysis flags initialized.")
print(f"ANALYSIS_OUTPUT_DIR={ANALYSIS_OUTPUT_DIR}")


## 1) Checklist Adopsi dari Repo Referensi

Cell ini merangkum step detail yang biasanya muncul di repo referensi, lalu dipetakan ke stage notebook ini.


In [ ]:
reference_steps = [
    ("Dataset inventory + schema check", "Section 2.0 - 2.2"),
    ("Duplicate / missing / inf diagnostics", "Section 2.3"),
    ("Label harmonization + feature intersection", "Section 2.3"),
    ("Deep dataset provenance, quality, drift, ablation", "Section 2.4 - 2.8"),
    ("Config review sebelum preprocess", "Section 3.0"),
    ("Preprocess execution + artifact verification", "Section 3.1 - 3.2"),
    ("Training diagnostics (history + checkpoints)", "Section 4.0 - 4.2"),
    ("Zero-shot vs few-shot evaluation", "Section 5.0 - 5.2"),
    ("Baseline comparison", "Section 6.0 - 6.1"),
    ("Research summary export", "Section 7.0"),
]

checklist_df = pd.DataFrame(reference_steps, columns=["reference_pattern", "implemented_in_notebook"])
display(checklist_df)


## 2) Data Discovery dan Pre-Evaluation Audit


In [ ]:
from scripts.utils import list_csv_files

cfg = load_config(str(BASE_CONFIG_PATH if BASE_CONFIG_PATH.exists() else Path("config.yaml")))
paths_cfg = cfg["paths"]

cic_dir = Path(paths_cfg["data_raw_cic"])
cse_dir = Path(paths_cfg["data_raw_cse"])

cic_files = list_csv_files(cic_dir)
cse_files = list_csv_files(cse_dir)


def file_inventory(files, dataset_name):
    rows = []
    total_size = 0
    for p in files:
        size = p.stat().st_size
        total_size += size
        rows.append(
            {
                "dataset": dataset_name,
                "file": p.name,
                "size_mb": round(size / (1024**2), 2),
            }
        )
    return rows, total_size

rows_cic, size_cic = file_inventory(cic_files, "CIC-IDS2017")
rows_cse, size_cse = file_inventory(cse_files, "CSE-CIC-IDS2018")
inventory_df = pd.DataFrame(rows_cic + rows_cse)

print(f"CIC files: {len(cic_files)} | total size: {size_cic/(1024**3):.2f} GB")
print(f"CSE files: {len(cse_files)} | total size: {size_cse/(1024**3):.2f} GB")
display(inventory_df)


### 2.1 Schema Snapshot dan Label Detection

Langkah ini memastikan kita tahu kolom label, kolom yang dibuang, dan perkiraan jumlah fitur mentah.


In [ ]:
from scripts.preprocess import load_header_columns
from scripts.utils import detect_label_column

label_candidates = cfg["preprocess"]["label_candidates"]
drop_columns = set(cfg["preprocess"]["drop_columns"])

summary_rows = []
for dataset_name, files in [("CIC", cic_files), ("CSE", cse_files)]:
    if not files:
        continue
    headers = load_header_columns(files[0])
    label_col = detect_label_column(headers, label_candidates)
    feature_candidates = [c for c in headers if c not in drop_columns and c != label_col]
    summary_rows.append(
        {
            "dataset": dataset_name,
            "sample_file": files[0].name,
            "total_columns": len(headers),
            "detected_label": label_col,
            "feature_candidates_after_drop": len(feature_candidates),
        }
    )

schema_df = pd.DataFrame(summary_rows)
display(schema_df)


### 2.2 Sample Quality Audit (Duplicate, Missing, Inf, Label Mix)

Audit ini mengimitasi langkah EDA dari repo referensi tanpa harus load semua data ke memory.


In [ ]:
def quick_quality_report(csv_path, label_col, sample_rows=50000):
    df = pd.read_csv(csv_path, nrows=sample_rows, skipinitialspace=True)
    df.columns = [str(c).strip() for c in df.columns]

    numeric_df = df.select_dtypes(include=[np.number])
    missing_cells = int(df.isna().sum().sum())
    inf_cells = int(np.isinf(numeric_df.to_numpy()).sum()) if numeric_df.shape[1] > 0 else 0

    row = {
        "file": csv_path.name,
        "rows_sampled": len(df),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": missing_cells,
        "inf_cells": inf_cells,
    }

    if label_col in df.columns:
        vc = df[label_col].astype(str).value_counts().head(5).to_dict()
        row["top_labels"] = vc
    else:
        row["top_labels"] = "label_not_found"

    return row

reports = []
for files, dataset_name in [(cic_files, "CIC"), (cse_files, "CSE")]:
    if not files:
        continue
    headers = load_header_columns(files[0])
    label_col = detect_label_column(headers, label_candidates)
    for p in files[:2]:
        rep = quick_quality_report(p, label_col=label_col, sample_rows=50000)
        rep["dataset"] = dataset_name
        reports.append(rep)

quality_df = pd.DataFrame(reports)
display(quality_df)


### 2.3 Feature Intersection dan Harmonisasi CIC vs CSE

Langkah ini mengikuti fungsi preprocessing project (`compute_feature_intersection`, `build_column_mapper`) untuk memastikan alignment sama dengan pipeline utama.


In [ ]:
from scripts.preprocess import compute_feature_intersection, build_column_mapper

cic_label_col, cic_features = compute_feature_intersection(cic_files, label_candidates, list(drop_columns))

cic_reference_cols = load_header_columns(cic_files[0]) if cic_files else []
cse_mapper = {}
for cse_path in cse_files:
    cse_cols = load_header_columns(cse_path)
    cse_mapper.update(build_column_mapper(cic_reference_cols, cse_cols))

cse_label_col, cse_features = compute_feature_intersection(
    cse_files,
    label_candidates,
    list(drop_columns),
    column_mapper=cse_mapper,
)

shared_features = sorted(set(cic_features).intersection(cse_features))

alignment_report = {
    "cic_label_col": cic_label_col,
    "cse_label_col": cse_label_col,
    "cic_feature_count": len(cic_features),
    "cse_feature_count": len(cse_features),
    "shared_feature_count": len(shared_features),
    "mapper_entries": len(cse_mapper),
}

print(json.dumps(alignment_report, indent=2))
print("Sample shared features:", shared_features[:12])


## 2.4 Dataset Provenance and Integrity

Audit provenance lengkap: inventory file, line count, quick signature, optional full hash,
dan schema mismatch lintas file.


In [ ]:
import hashlib
from sklearn.feature_selection import mutual_info_classif

np.random.seed(ANALYSIS_RANDOM_SEED)


def analysis_root_for_run(run_id: str) -> Path:
    out = Path("reports") / "research" / run_id / "analysis"
    out.mkdir(parents=True, exist_ok=True)
    return out


ANALYSIS_OUTPUT_DIR = analysis_root_for_run(INSPECT_RUN_ID)


def save_json_report(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def save_csv_report(df: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


def _count_lines(path: Path) -> int:
    if "count_file_lines" in globals():
        return int(count_file_lines(path))
    total = 0
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            total += chunk.count(b"\n")
    return int(total)


def quick_file_signature(path: Path) -> str:
    h = hashlib.sha256()
    size = path.stat().st_size
    with path.open("rb") as f:
        h.update(f.read(1024 * 1024))
        if size > 1024 * 1024:
            f.seek(max(size - 1024 * 1024, 0))
            h.update(f.read(1024 * 1024))
    h.update(str(size).encode("utf-8"))
    return h.hexdigest()


def full_file_hash(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


cic_files_for_audit = cic_files[:ANALYSIS_MAX_FILES_PER_DATASET] if ANALYSIS_MAX_FILES_PER_DATASET else cic_files
cse_files_for_audit = cse_files[:ANALYSIS_MAX_FILES_PER_DATASET] if ANALYSIS_MAX_FILES_PER_DATASET else cse_files

inv_rows = []
mismatch_rows = []
for ds, files in [("CIC-IDS2017", cic_files_for_audit), ("CSE-CIC-IDS2018", cse_files_for_audit)]:
    if not files:
        continue
    ref_cols = [str(c).strip() for c in load_header_columns(files[0])]
    ref_set = set(ref_cols)
    for p in files:
        cols = [str(c).strip() for c in load_header_columns(p)]
        row = {
            "dataset": ds,
            "file": p.name,
            "size_mb": round(p.stat().st_size / (1024 ** 2), 2),
            "line_count": _count_lines(p),
            "quick_signature_sha256": quick_file_signature(p),
            "n_columns": len(cols),
            "label_col": detect_label_column(cols, label_candidates),
        }
        if ANALYSIS_COMPUTE_FULL_HASH:
            row["full_sha256"] = full_file_hash(p)
        inv_rows.append(row)

        col_set = set(cols)
        miss = sorted(ref_set - col_set)
        extra = sorted(col_set - ref_set)
        if miss or extra:
            mismatch_rows.append(
                {
                    "dataset": ds,
                    "file": p.name,
                    "missing_count": len(miss),
                    "extra_count": len(extra),
                    "missing_cols": ";".join(miss),
                    "extra_cols": ";".join(extra),
                }
            )

dataset_inventory_df = pd.DataFrame(inv_rows)
schema_mismatch_df = pd.DataFrame(mismatch_rows)
display(dataset_inventory_df)
if not schema_mismatch_df.empty:
    display(schema_mismatch_df)

save_csv_report(dataset_inventory_df, ANALYSIS_OUTPUT_DIR / "dataset_inventory.csv")
save_csv_report(schema_mismatch_df, ANALYSIS_OUTPUT_DIR / "schema_mismatch_report.csv")
save_json_report(
    {
        "sampled_cic_files": len(cic_files_for_audit),
        "sampled_cse_files": len(cse_files_for_audit),
        "compute_full_hash": ANALYSIS_COMPUTE_FULL_HASH,
        "inventory_rows": int(len(dataset_inventory_df)),
        "mismatch_rows": int(len(schema_mismatch_df)),
    },
    ANALYSIS_OUTPUT_DIR / "dataset_integrity.json",
)

print("Saved integrity artifacts to", ANALYSIS_OUTPUT_DIR)


## 2.5 Label Distribution and Data Quality Deep Dive

Profil label dan kualitas data pada sampel besar per file:
duplicate, missing, inf, rasio benign/attack, serta quantile fitur.


In [ ]:
def normalize_binary_label(x: str) -> int:
    return 0 if str(x).strip().upper() == "BENIGN" else 1


def sample_file_stats(path: Path, label_col: str, nrows: int):
    df = pd.read_csv(path, nrows=nrows, skipinitialspace=True, low_memory=False)
    df.columns = [str(c).strip() for c in df.columns]
    num = df.select_dtypes(include=[np.number])
    inf_cells = int(np.isinf(num.to_numpy()).sum()) if not num.empty else 0
    missing_cells = int(df.isna().sum().sum())
    dup_rows = int(df.duplicated().sum())

    out = {
        "file": path.name,
        "rows_sampled": int(len(df)),
        "duplicate_rows": dup_rows,
        "missing_cells": missing_cells,
        "inf_cells": inf_cells,
    }

    labels = []
    if label_col in df.columns:
        raw_vc = df[label_col].astype(str).value_counts(dropna=False)
        mapped = df[label_col].astype(str).map(normalize_binary_label)
        bin_vc = mapped.value_counts(dropna=False).to_dict()
        out["benign_count"] = int(bin_vc.get(0, 0))
        out["attack_count"] = int(bin_vc.get(1, 0))
        for label_name, n in raw_vc.items():
            labels.append({"file": path.name, "raw_label": str(label_name), "count": int(n)})
    else:
        out["benign_count"] = None
        out["attack_count"] = None

    return out, labels


quality_rows = []
label_rows = []
for ds, files in [("CIC-IDS2017", cic_files_for_audit), ("CSE-CIC-IDS2018", cse_files_for_audit)]:
    if not files:
        continue
    label_col = detect_label_column(load_header_columns(files[0]), label_candidates)
    for p in files:
        q, labels = sample_file_stats(p, label_col, ANALYSIS_SAMPLE_ROWS_PER_FILE)
        q["dataset"] = ds
        quality_rows.append(q)
        for item in labels:
            item["dataset"] = ds
            label_rows.append(item)

quality_deep_df = pd.DataFrame(quality_rows)
if not quality_deep_df.empty:
    quality_deep_df["benign_attack_ratio"] = quality_deep_df.apply(
        lambda r: None if not r.get("attack_count") else round((r.get("benign_count") or 0) / r["attack_count"], 4),
        axis=1,
    )
label_by_file_df = pd.DataFrame(label_rows)
label_overall_df = (
    label_by_file_df.groupby(["dataset", "raw_label"], as_index=False)["count"].sum()
    if not label_by_file_df.empty
    else pd.DataFrame(columns=["dataset", "raw_label", "count"])
)

# Quantile snapshot (top 30 shared features, first 2 files each dataset)
quant_rows = []
probe_features = shared_features[: min(30, len(shared_features))]
for ds, files in [("CIC-IDS2017", cic_files_for_audit[:2]), ("CSE-CIC-IDS2018", cse_files_for_audit[:2])]:
    for p in files:
        raw_cols = pd.read_csv(p, nrows=0, low_memory=False, skipinitialspace=True).columns.tolist()
        raw_to_clean = {str(c).strip(): c for c in raw_cols}
        usecols_clean = [c for c in probe_features if c in raw_to_clean]
        usecols_raw = [raw_to_clean[c] for c in usecols_clean]

        if not usecols_raw:
            continue

        df = pd.read_csv(
            p,
            usecols=usecols_raw,
            nrows=min(ANALYSIS_SAMPLE_ROWS_PER_FILE, 50000),
            low_memory=False,
            skipinitialspace=True,
        )
        df.columns = [str(c).strip() for c in df.columns]

        for col in usecols_clean:
            if col not in df.columns:
                continue
            s = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
            if s.empty:
                continue
            q = s.quantile([0.01, 0.05, 0.5, 0.95, 0.99]).to_dict()
            quant_rows.append(
                {
                    "dataset": ds,
                    "file": p.name,
                    "feature": col,
                    "q01": float(q.get(0.01, np.nan)),
                    "q05": float(q.get(0.05, np.nan)),
                    "q50": float(q.get(0.5, np.nan)),
                    "q95": float(q.get(0.95, np.nan)),
                    "q99": float(q.get(0.99, np.nan)),
                }
            )
feature_quantiles_df = pd.DataFrame(quant_rows)

display(quality_deep_df)
display(label_overall_df)

save_csv_report(quality_deep_df, ANALYSIS_OUTPUT_DIR / "data_quality_by_file.csv")
save_csv_report(label_by_file_df, ANALYSIS_OUTPUT_DIR / "label_distribution_by_file.csv")
save_csv_report(label_overall_df, ANALYSIS_OUTPUT_DIR / "label_distribution_overall.csv")
save_csv_report(feature_quantiles_df, ANALYSIS_OUTPUT_DIR / "feature_quantiles.csv")
print("Saved quality artifacts.")


## 2.6 Cross-Dataset Shift and Feature Engineering Analysis

Statistik FE yang ditambahkan: variance + NZV flag, high-correlation pairs,
mutual information, dan drift score (PSI + KS).


In [ ]:
def build_feature_sample(files, label_col, features, max_rows, chunk_rows=50000):
    x_parts = []
    y_parts = []
    remaining = int(max_rows)
    for p in files:
        if remaining <= 0:
            break
        take = min(chunk_rows, remaining)

        raw_cols = pd.read_csv(p, nrows=0, low_memory=False, skipinitialspace=True).columns.tolist()
        raw_to_clean = {str(c).strip(): c for c in raw_cols}

        feats_clean = [f for f in features if f in raw_to_clean]
        if not feats_clean or label_col not in raw_to_clean:
            continue

        usecols_raw = [raw_to_clean[f] for f in feats_clean] + [raw_to_clean[label_col]]
        df = pd.read_csv(p, usecols=usecols_raw, nrows=take, skipinitialspace=True, low_memory=False)
        df.columns = [str(c).strip() for c in df.columns]

        feats_final = [f for f in feats_clean if f in df.columns]
        if label_col not in df.columns or not feats_final:
            continue

        x = df[feats_final].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
        y = df[label_col].astype(str).map(normalize_binary_label).fillna(1).astype(np.int32)
        if not x.empty:
            x_parts.append(x)
            y_parts.append(y)
            remaining -= len(x)

    if not x_parts:
        return pd.DataFrame(columns=features), pd.Series(dtype=np.int32)

    x_all = pd.concat(x_parts, ignore_index=True)
    for f in features:
        if f not in x_all.columns:
            x_all[f] = 0.0
    x_all = x_all[features]

    return x_all, pd.concat(y_parts, ignore_index=True)


def compute_psi(expected: np.ndarray, actual: np.ndarray, bins=10, eps=1e-8) -> float:
    expected = expected[np.isfinite(expected)]
    actual = actual[np.isfinite(actual)]
    if expected.size == 0 or actual.size == 0:
        return np.nan
    edges = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0
    e_hist, _ = np.histogram(expected, bins=edges)
    a_hist, _ = np.histogram(actual, bins=edges)
    e_ratio = np.clip(e_hist / max(e_hist.sum(), 1), eps, None)
    a_ratio = np.clip(a_hist / max(a_hist.sum(), 1), eps, None)
    return float(np.sum((a_ratio - e_ratio) * np.log(a_ratio / e_ratio)))


def compute_ks_stat(expected: np.ndarray, actual: np.ndarray) -> float:
    expected = np.sort(expected[np.isfinite(expected)])
    actual = np.sort(actual[np.isfinite(actual)])
    if expected.size == 0 or actual.size == 0:
        return np.nan
    values = np.sort(np.unique(np.concatenate([expected, actual])))
    cdf_e = np.searchsorted(expected, values, side="right") / expected.size
    cdf_a = np.searchsorted(actual, values, side="right") / actual.size
    return float(np.max(np.abs(cdf_e - cdf_a)))


if not RUN_DEEP_AUDIT:
    print("RUN_DEEP_AUDIT=False -> skip deep feature analysis.")
else:
    cic_x_sample, cic_y_sample = build_feature_sample(cic_files_for_audit, cic_label_col, shared_features, ANALYSIS_SAMPLE_ROWS_PER_FILE)
    cse_x_sample, cse_y_sample = build_feature_sample(cse_files_for_audit, cse_label_col, shared_features, ANALYSIS_SAMPLE_ROWS_PER_FILE)

    if cic_x_sample.empty or cse_x_sample.empty:
        print("Insufficient sampled rows for deep FE analysis.")
    else:
        feature_stats_df = pd.DataFrame({"feature": shared_features})
        feature_stats_df["variance_cic"] = [float(cic_x_sample[f].var()) for f in shared_features]
        feature_stats_df["nzv_flag"] = feature_stats_df["variance_cic"] <= 1e-6

        # correlation pairs
        corr = cic_x_sample[shared_features].corr().replace([np.inf, -np.inf], np.nan)
        corr_rows = []
        cols = list(corr.columns)
        for i in range(len(cols)):
            for j in range(i + 1, len(cols)):
                v = corr.iloc[i, j]
                if pd.notna(v) and abs(float(v)) >= 0.95:
                    corr_rows.append({"feature_a": cols[i], "feature_b": cols[j], "corr": float(v)})
        feature_corr_pairs_df = pd.DataFrame(corr_rows)

        # MI
        try:
            mi_vals = mutual_info_classif(
                cic_x_sample[shared_features].to_numpy(),
                cic_y_sample.to_numpy(),
                random_state=ANALYSIS_RANDOM_SEED,
                discrete_features=False,
            )
        except Exception:
            mi_vals = np.full(len(shared_features), np.nan)
        feature_stats_df["mutual_info"] = mi_vals

        # Drift
        drift_rows = []
        if RUN_DRIFT_ANALYSIS:
            for f in shared_features:
                e = cic_x_sample[f].to_numpy(dtype=float)
                a = cse_x_sample[f].to_numpy(dtype=float)
                drift_rows.append(
                    {
                        "feature": f,
                        "psi": compute_psi(e, a),
                        "ks_stat": compute_ks_stat(e, a),
                        "mean_cic": float(np.nanmean(e)),
                        "mean_cse": float(np.nanmean(a)),
                    }
                )
        drift_df = pd.DataFrame(drift_rows)
        if not drift_df.empty:
            feature_stats_df = feature_stats_df.merge(drift_df[["feature", "psi", "ks_stat"]], on="feature", how="left")

        display(feature_stats_df.sort_values(by=["mutual_info", "psi"], ascending=[False, True]).head(20))
        if not drift_df.empty:
            display(drift_df.sort_values("psi", ascending=False).head(20))

        save_csv_report(feature_stats_df, ANALYSIS_OUTPUT_DIR / "feature_rank_mi.csv")
        save_csv_report(feature_corr_pairs_df, ANALYSIS_OUTPUT_DIR / "feature_corr_pairs.csv")
        save_csv_report(drift_df, ANALYSIS_OUTPUT_DIR / "drift_report.csv")
        save_json_report(
            {
                "sample_rows_cic": int(len(cic_x_sample)),
                "sample_rows_cse": int(len(cse_x_sample)),
                "shared_feature_count": int(len(shared_features)),
                "high_corr_pairs": int(len(feature_corr_pairs_df)),
                "drift_enabled": bool(RUN_DRIFT_ANALYSIS),
            },
            ANALYSIS_OUTPUT_DIR / "feature_analysis_report.json",
        )
        print("Saved feature engineering artifacts.")


## 2.7 Mini Baseline Importance and Ablation

Eksperimen cepat untuk memahami subset fitur:
`all_shared`, `filtered_nzv`, `top_mi`, dan `drift_stable`.


In [ ]:
if not RUN_BASELINE_IMPORTANCE_ANALYSIS:
    print("RUN_BASELINE_IMPORTANCE_ANALYSIS=False -> skip baseline mini ablation.")
elif "feature_stats_df" not in globals() or "cic_x_sample" not in globals() or cic_x_sample.empty:
    print("Feature analysis outputs not ready. Run section 2.6 first.")
else:
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.inspection import permutation_importance
    from sklearn.metrics import f1_score, precision_score, recall_score

    top_k = max(5, int(ANALYSIS_FEATURE_TOP_K))
    all_set = list(shared_features)
    nzv_set = feature_stats_df.loc[~feature_stats_df["nzv_flag"], "feature"].tolist()
    mi_set = feature_stats_df.sort_values("mutual_info", ascending=False)["feature"].head(top_k).tolist()
    if "psi" in feature_stats_df.columns and feature_stats_df["psi"].notna().any():
        stable_set = feature_stats_df.sort_values("psi", ascending=True)["feature"].head(top_k).tolist()
    else:
        stable_set = all_set[:top_k]

    feature_sets = {
        "all_shared": all_set,
        "filtered_nzv": nzv_set,
        "top_mi": mi_set,
        "drift_stable": stable_set,
    }

    ablation_rows = []
    for name, feats in feature_sets.items():
        feats = [f for f in feats if f in cic_x_sample.columns and f in cse_x_sample.columns]
        if len(feats) < 3:
            continue
        clf = RandomForestClassifier(
            n_estimators=250,
            random_state=ANALYSIS_RANDOM_SEED,
            n_jobs=-1,
            class_weight="balanced_subsample",
        )
        clf.fit(cic_x_sample[feats], cic_y_sample)
        pred = clf.predict(cse_x_sample[feats])
        ablation_rows.append(
            {
                "subset": name,
                "n_features": len(feats),
                "f1": float(f1_score(cse_y_sample, pred, zero_division=0)),
                "precision": float(precision_score(cse_y_sample, pred, zero_division=0)),
                "recall": float(recall_score(cse_y_sample, pred, zero_division=0)),
            }
        )

    ablation_results_df = pd.DataFrame(ablation_rows).sort_values(by="f1", ascending=False)
    display(ablation_results_df)
    save_csv_report(ablation_results_df, ANALYSIS_OUTPUT_DIR / "ablation_results.csv")

    if not ablation_results_df.empty:
        best = str(ablation_results_df.iloc[0]["subset"])
        best_feats = [f for f in feature_sets[best] if f in cic_x_sample.columns and f in cse_x_sample.columns]
        best_clf = RandomForestClassifier(
            n_estimators=300,
            random_state=ANALYSIS_RANDOM_SEED,
            n_jobs=-1,
            class_weight="balanced_subsample",
        )
        best_clf.fit(cic_x_sample[best_feats], cic_y_sample)
        perm = permutation_importance(
            best_clf,
            cse_x_sample[best_feats],
            cse_y_sample,
            n_repeats=5,
            random_state=ANALYSIS_RANDOM_SEED,
            n_jobs=-1,
            scoring="f1",
        )
        baseline_importance_df = pd.DataFrame(
            {
                "feature": best_feats,
                "importance_mean": perm.importances_mean,
                "importance_std": perm.importances_std,
            }
        ).sort_values("importance_mean", ascending=False)
        display(baseline_importance_df.head(20))
        save_csv_report(baseline_importance_df, ANALYSIS_OUTPUT_DIR / "baseline_importance.csv")
        (ANALYSIS_OUTPUT_DIR / "baseline_model_card.md").write_text(
            "\n".join(
                [
                    "# Baseline Model Card",
                    "",
                    "- model: RandomForestClassifier",
                    f"- selected_subset: {best}",
                    f"- n_features: {len(best_feats)}",
                    "",
                    "## Top 10 Features",
                    baseline_importance_df.head(10).to_markdown(index=False),
                ]
            )
            + "\n",
            encoding="utf-8",
        )

    print("Saved ablation and baseline importance artifacts.")


## 2.8 Research Readiness Notes and Limitations

Ringkasan risiko metodologis dan checklist artifact agar siap dipakai untuk laporan riset.


In [ ]:
notes_path = ANALYSIS_OUTPUT_DIR / "research_notes.md"
notes_path.write_text(
    "\n".join(
        [
            "# Research Readiness Notes",
            "",
            "## Strengths",
            "- Pipeline sudah end-to-end dan terstruktur.",
            "- Ada audit dataset, drift, dan feature engineering diagnostics.",
            "",
            "## Limitations",
            "- CSE-CIC-IDS2018 masih dari Kaggle mirror.",
            "- Domain shift 2017 -> 2018 perlu mitigasi threshold/calibration.",
            "",
            "## Next Steps",
            "1. Cross-day validation.",
            "2. Calibration per attack family.",
            "3. Sensitivity study feature filter + scale guard.",
        ]
    )
    + "\n",
    encoding="utf-8",
)

expected = [
    "dataset_inventory.csv",
    "schema_mismatch_report.csv",
    "dataset_integrity.json",
    "data_quality_by_file.csv",
    "label_distribution_by_file.csv",
    "label_distribution_overall.csv",
    "feature_quantiles.csv",
    "feature_rank_mi.csv",
    "feature_corr_pairs.csv",
    "drift_report.csv",
    "feature_analysis_report.json",
    "ablation_results.csv",
    "baseline_importance.csv",
    "baseline_model_card.md",
    "research_notes.md",
]
rows = []
for name in expected:
    p = ANALYSIS_OUTPUT_DIR / name
    rows.append({"artifact": name, "exists": p.exists(), "path": str(p)})
persist_df = pd.DataFrame(rows)
display(persist_df)
save_json_report(
    {
        "analysis_output_dir": str(ANALYSIS_OUTPUT_DIR),
        "existing_artifacts": int(persist_df["exists"].sum()),
        "total_artifacts": int(len(persist_df)),
    },
    ANALYSIS_OUTPUT_DIR / "persist_check.json",
)
print("Saved:", notes_path)


## 3) Preprocessing Stage (Granular)


In [ ]:
cfg_path = ensure_generated_config(RUN_ID_HYBRID_ZERO)
cfg = load_config(str(cfg_path))
pre_cfg = cfg["preprocess"]

pre_view = {
    "run_id": RUN_ID_HYBRID_ZERO,
    "generated_config": str(cfg_path),
    "window_size": pre_cfg.get("window_size"),
    "stride": pre_cfg.get("stride"),
    "scaler": pre_cfg.get("scaler"),
    "scaler_fit_mode": pre_cfg.get("scaler_fit_mode"),
    "chunksize": pre_cfg.get("chunksize"),
    "shard_enable": pre_cfg.get("shard_enable"),
    "shard_size": pre_cfg.get("shard_size"),
    "split_by_file": pre_cfg.get("split_by_file"),
    "feature_filter": pre_cfg.get("feature_filter"),
    "scale_guard": pre_cfg.get("scale_guard"),
}
print(json.dumps(pre_view, indent=2))


### 3.1 Execute Preprocess Pipeline

Script preprocess dipanggil dengan generated config per-run:
`research/sprint2/generated_configs/<run_id>.yaml`



In [ ]:
if RUN_PREPROCESS:
    cfg_path = ensure_generated_config(RUN_ID_HYBRID_ZERO)
    run_cmd_stream([sys.executable, "scripts/preprocess.py", "--config", str(cfg_path)], cwd=PROJECT_ROOT)
else:
    print("RUN_PREPROCESS=False -> skip preprocess execution.")


### 3.2 Verify Preprocess Artifacts

Validasi manifest shard, scaler, feature metadata, dan report optional.


In [ ]:
data_source_run_id = resolve_data_source_run(INSPECT_RUN_ID)
processed_dir = Path("data") / "research" / data_source_run_id / "processed"
shard_root = processed_dir / "shards"

artifact_paths = {
    "scaler": processed_dir / "scaler.pkl",
    "feature_columns": processed_dir / "feature_columns.json",
    "feature_filter_report": processed_dir / "feature_filter_report.json",
    "scale_guard_report": processed_dir / "scale_guard_report.json",
    "cic_train_manifest": shard_root / "cic" / "train" / "manifest.json",
    "cic_val_manifest": shard_root / "cic" / "val" / "manifest.json",
    "cic_test_manifest": shard_root / "cic" / "test" / "manifest.json",
    "cse_test_manifest": shard_root / "cse" / "test" / "manifest.json",
}

artifact_df = pd.DataFrame(
    [{"artifact": k, "path": str(v), "exists": v.exists()} for k, v in artifact_paths.items()]
)
print(f"Inspect run_id={INSPECT_RUN_ID} | data_source_run_id={data_source_run_id}")
display(artifact_df)

manifest_keys = ["cic_train_manifest", "cic_val_manifest", "cic_test_manifest", "cse_test_manifest"]
manifest_rows = []
for key in manifest_keys:
    m = read_json(artifact_paths[key])
    if m is None:
        continue
    manifest_rows.append(
        {
            "manifest": key,
            "total_samples": m.get("total_samples"),
            "num_shards": m.get("num_shards"),
            "input_shape": m.get("input_shape"),
        }
    )

if manifest_rows:
    display(pd.DataFrame(manifest_rows))
else:
    print("Manifest belum tersedia. Jalankan preprocess dulu.")


### 3.3 Leakage Guard Quick Checks

Validasi cepat terhadap shard split:
- train/val manifest harus ada dan jumlah sample > 0
- test shard harus punya label `y`
- train/val shard memang tanpa `y` (sesuai desain unsupervised training)


In [ ]:
issues = []

train_manifest = read_json(artifact_paths["cic_train_manifest"])
val_manifest = read_json(artifact_paths["cic_val_manifest"])
cic_test_manifest = read_json(artifact_paths["cic_test_manifest"])

if train_manifest is None or train_manifest.get("total_samples", 0) <= 0:
    issues.append("train manifest missing or empty")
if val_manifest is None or val_manifest.get("total_samples", 0) <= 0:
    issues.append("val manifest missing or empty")


def inspect_npz_keys(manifest_obj, label):
    if not manifest_obj or not manifest_obj.get("shards"):
        return None
    first_rel = manifest_obj["shards"][0]["path"]
    npz_path = shard_root / first_rel
    arr = np.load(npz_path)
    return label, list(arr.keys()), {k: arr[k].shape for k in arr.files}

checks = []
for name, mani in [
    ("train", train_manifest),
    ("val", val_manifest),
    ("cic_test", cic_test_manifest),
]:
    out = inspect_npz_keys(mani, name)
    if out is not None:
        checks.append(out)

for label, keys, shapes in checks:
    print(f"{label}: keys={keys}")
    print(f"{label}: shapes={shapes}")

if issues:
    print("Issues:")
    for x in issues:
        print("-", x)
else:
    print("Leakage guard quick checks passed.")


## 4) Hybrid CNN-LSTM AE Training


In [ ]:
cfg_path = ensure_generated_config(RUN_ID_HYBRID_ZERO)
cfg = load_config(str(cfg_path))
train_cfg = cfg["training"]

train_view = {
    "run_id": RUN_ID_HYBRID_ZERO,
    "generated_config": str(cfg_path),
    "epochs": train_cfg.get("epochs"),
    "batch_size": train_cfg.get("batch_size"),
    "learning_rate": train_cfg.get("learning_rate"),
    "early_stopping_patience": train_cfg.get("early_stopping_patience"),
    "lr_scheduler": train_cfg.get("lr_scheduler"),
    "cnn_filters": train_cfg.get("cnn_filters"),
    "lstm_units": train_cfg.get("lstm_units"),
    "latent_dim": train_cfg.get("latent_dim"),
    "dropout": train_cfg.get("dropout"),
}
print(json.dumps(train_view, indent=2))


In [ ]:
if RUN_TRAIN_HYBRID:
    cfg_path = ensure_generated_config(RUN_ID_HYBRID_ZERO)
    run_cmd_stream([sys.executable, "scripts/train_cnn_lstm_ae.py", "--config", str(cfg_path)], cwd=PROJECT_ROOT)
else:
    print("RUN_TRAIN_HYBRID=False -> skip hybrid training.")


### 4.1 Training History Inspection

Membaca `results/logs/cnn_lstm_history.json` untuk melihat konvergensi.


In [ ]:
history_path = Path("results") / "research" / RUN_ID_HYBRID_ZERO / "logs" / "cnn_lstm_history.json"
history = read_json(history_path)

if history is None:
    print(f"History file not found: {history_path}")
else:
    hist_df = pd.DataFrame(history)
    display(hist_df.tail())

    plot_cols = [c for c in ["loss", "val_loss"] if c in hist_df.columns]
    if plot_cols:
        ax = hist_df[plot_cols].plot(figsize=(10, 4), title="Hybrid Training Loss Curves")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("MSE Loss")
        plt.show()


## 5) Evaluation: Zero-Shot vs Few-Shot


In [ ]:
print("Eval helper ready. Use run_eval_for_run_id(run_id).")


In [ ]:
if RUN_EVAL_ZERO_SHOT:
    run_eval_for_run_id(RUN_ID_HYBRID_ZERO)
else:
    print("RUN_EVAL_ZERO_SHOT=False -> skip hybrid zero-shot eval.")

if RUN_EVAL_FEW_SHOT:
    run_eval_for_run_id(RUN_ID_HYBRID_FEW)
else:
    print("RUN_EVAL_FEW_SHOT=False -> skip hybrid few-shot eval.")


### 5.1 Compare Hybrid Evaluation Variants


In [ ]:
rows = []
for run_id, label in [
    (RUN_ID_HYBRID_ZERO, "hybrid_zero_shot"),
    (RUN_ID_HYBRID_FEW, "hybrid_few_shot"),
]:
    row = collect_metric_row(run_id, label)
    if row:
        rows.append(row)

if not rows:
    print("No hybrid metric files found yet.")
else:
    hybrid_df = pd.DataFrame(rows)
    display(hybrid_df)

    plot_df = hybrid_df[["variant", "cic_f1", "cse_f1"]].set_index("variant")
    plot_df.plot(kind="bar", figsize=(9, 4), title="Hybrid F1 Comparison (CIC vs CSE)")
    plt.ylabel("F1")
    plt.ylim(0, 1)
    plt.show()


## 6) Baseline LSTM Autoencoder + Comparison


In [ ]:
if RUN_TRAIN_BASELINE:
    cfg_path = ensure_generated_config(RUN_ID_BASELINE_ZERO)
    run_cmd_stream([sys.executable, "scripts/train_lstm_ae.py", "--config", str(cfg_path)], cwd=PROJECT_ROOT)
else:
    print("RUN_TRAIN_BASELINE=False -> skip baseline training.")

if RUN_EVAL_BASELINE_ZERO_SHOT:
    run_eval_for_run_id(RUN_ID_BASELINE_ZERO)
else:
    print("RUN_EVAL_BASELINE_ZERO_SHOT=False -> skip baseline zero-shot eval.")

if RUN_EVAL_BASELINE_FEW_SHOT:
    run_eval_for_run_id(RUN_ID_BASELINE_FEW)
else:
    print("RUN_EVAL_BASELINE_FEW_SHOT=False -> skip baseline few-shot eval.")


### 6.1 Hybrid vs Baseline Table


In [ ]:
compare_rows = []
for run_id, label in [
    (RUN_ID_HYBRID_ZERO, "hybrid_zero_shot"),
    (RUN_ID_HYBRID_FEW, "hybrid_few_shot"),
    (RUN_ID_BASELINE_ZERO, "baseline_zero_shot"),
    (RUN_ID_BASELINE_FEW, "baseline_few_shot"),
]:
    row = collect_metric_row(run_id, label)
    if row:
        compare_rows.append(row)

if not compare_rows:
    print("No metrics available for comparison yet.")
else:
    compare_df = pd.DataFrame(compare_rows)
    display(compare_df.sort_values(by=["variant"]))

    view_cols = ["variant", "cic_f1", "cse_f1", "f1_gap", "cic_auc", "cse_auc"]
    plot_ready = compare_df[view_cols].copy()
    display(plot_ready)


## 7) Export Ringkasan Eksekusi Notebook

Cell ini membuat ringkasan markdown yang bisa langsung dipakai untuk dokumentasi sprint/report.


In [ ]:
summary_path = Path("results/research/sprint2/notebook_execution_summary.md")
summary_path.parent.mkdir(parents=True, exist_ok=True)

all_rows = []
for run_id, label in [
    (RUN_ID_HYBRID_ZERO, "hybrid_zero_shot"),
    (RUN_ID_HYBRID_FEW, "hybrid_few_shot"),
    (RUN_ID_BASELINE_ZERO, "baseline_zero_shot"),
    (RUN_ID_BASELINE_FEW, "baseline_few_shot"),
]:
    row = collect_metric_row(run_id, label)
    if row:
        all_rows.append(row)

lines = [
    "# Notebook Execution Summary",
    "",
    "Generated by `notebooks/research_master_comprehensive.ipynb`.",
    "",
    "## Available Metrics",
]

if not all_rows:
    lines.append("No metrics JSON files found yet. Run evaluation cells first.")
else:
    table_df = pd.DataFrame(all_rows)
    lines.append(table_df.to_markdown(index=False))

summary_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"Summary written to: {summary_path}")


## 8) Optional Sprint2 Summary Refresh

Aktifkan `RUN_SUMMARY_ONLY=True` jika ingin refresh agregasi sprint tanpa menjalankan preprocess/train/eval.


In [ ]:
if RUN_SUMMARY_ONLY:
    run_cmd_stream(
        [
            sys.executable,
            "scripts/research_sprint2.py",
            "--registry",
            str(REGISTRY_PATH),
            "--base-config",
            str(BASE_CONFIG_PATH),
            "--summarize-only",
        ],
        cwd=PROJECT_ROOT,
    )
else:
    print("RUN_SUMMARY_ONLY=False -> skip sprint summary refresh.")
